# Bike Count Estimation — Münster Hourly Time-Series Challenge

**Objective:** Predict hourly bike counts using temporal, weather, and lag features across two forecast horizons:
- **Horizon 1:** Next hour ($t+1$)
- **Horizon 2:** Next 24 hours ($t+1$ to $t+24$)

**Metric:** Mean Squared Error (MSE)

**Models:** Linear (Ridge), Tree-Based (LightGBM), Neural Network (PyTorch MLP/LSTM)

## 1. Environment Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import re
import warnings
from pathlib import Path

# Scikit-learn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.multioutput import MultiOutputRegressor

# Tree-based
import lightgbm as lgb

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")

PyTorch device: cpu


## 2. Data Loading & Initial Exploration

In [2]:
# --- CONFIGURATION ---
# Update this path to point to your training CSV/Excel file
TRAIN_DATA_PATH = Path("challenge_public_dataset.xlsx")

# Load data
if TRAIN_DATA_PATH.suffix == ".xlsx":
    df_raw = pd.read_excel(TRAIN_DATA_PATH)
else:
    df_raw = pd.read_csv(TRAIN_DATA_PATH)

print(f"Dataset shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(10)

Dataset shape: (8760, 10)
Columns: ['Month', 'Day', 'Hour', 'Weekday', 'Weather', 'Temperature (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)', 'BikeCount']


,Month,Day,Hour,Weekday,Weather,Temperature (°C),Humidity (%),Rain (mm),Wind (km/h),BikeCount
0,1,1,0,6,Sunny,14.0,61.0,0.0,34.0,73.0
1,1,1,1,6,Sunny,14.0,59.0,0.0,34.0,193.0
2,1,1,2,6,Partly Cloudy,14.0,57.0,0.0,33.0,240.0
3,1,1,3,6,Partly Cloudy,14.0,55.0,0.0,33.0,279.0
4,1,1,4,6,Partly Cloudy,14.0,56.0,0.0,33.0,194.0
5,1,1,5,6,Partly Cloudy,14.0,57.0,0.0,33.0,112.0
6,1,1,6,6,Partly Cloudy,14.0,58.0,0.0,33.0,45.0
7,1,1,7,6,Partly Cloudy,14.0,61.0,0.0,31.0,21.0
8,1,1,8,6,Partly Cloudy,13.0,64.0,0.0,29.0,26.0
9,1,1,9,6,Partly Cloudy,13.0,66.0,0.0,26.0,24.0


In [3]:
df_raw.info()
print("\n--- Descriptive Statistics ---")
df_raw.describe()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Month             8760 non-null   int64  
 1   Day               8760 non-null   int64  
 2   Hour              8760 non-null   int64  
 3   Weekday           8760 non-null   int64  
 4   Weather           8760 non-null   str    
 5   Temperature (°C)  8759 non-null   float64
 6   Humidity (%)      8759 non-null   float64
 7   Rain (mm)         8759 non-null   float64
 8   Wind (km/h)       8759 non-null   float64
 9   BikeCount         8759 non-null   float64
dtypes: float64(5), int64(4), str(1)
memory usage: 684.5 KB

--- Descriptive Statistics ---


,Month,Day,Hour,Weekday,Temperature (°C),Humidity (%),Rain (mm),Wind (km/h),BikeCount
count,8760.000000,8760.000000,8760.000000,8760.000000,8759.000000,8759.000000,8759.000000,8759.000000,8759.000000
mean,6.526027,15.720548,11.500114,3.008219,11.302774,77.102980,0.111622,15.889143,452.342048
std,3.448048,8.796749,6.922433,2.003519,7.138446,15.067433,0.462491,8.505827,363.804893
min,1.000000,1.000000,0.000000,0.000000,-7.000000,25.000000,0.000000,0.000000,0.000000
25%,4.000000,8.000000,5.750000,1.000000,6.000000,68.000000,0.000000,9.000000,119.000000
50%,7.000000,16.000000,11.500000,3.000000,11.000000,80.000000,0.000000,14.000000,380.000000
75%,10.000000,23.000000,17.250000,5.000000,16.000000,89.000000,0.000000,21.500000,730.000000
max,12.000000,31.000000,23.000000,6.000000,36.000000,100.000000,15.800000,48.000000,1789.000000


In [4]:
# Inspect unique values in potentially mixed-text columns
print("Unique 'Weekday' values:")
print(df_raw["Weekday"].unique())
print(f"\nUnique 'Weather' values:")
print(df_raw["Weather"].unique())

Unique 'Weekday' values:
[6 0 1 2 3 4 5]

Unique 'Weather' values:
<StringArray>
[                                        'Sunny',
                                 'Partly Cloudy',
                                      'Overcast',
                               'Occasional Rain',
                                       'Drizzle',
                                    'Light Rain',
                                  'Light Shower',
                                     'Light Fog',
                                        'Cloudy',
    'Occasional Thunderstorms and Precipitation',
                         'Occasional Light Rain',
                            'Occasional Drizzle',
 'Moderate to Heavy Snowfall with Thunderstorms',
                             'Moderate Snowfall',
                                    'Snowdrifts',
                    'Moderate to Heavy Snowfall',
                                     'Snowstorm',
                                'Heavy Snowfall',
                   

## 3. Advanced Feature Engineering Pipeline

This section encapsulates all transformations into reproducible functions that can be applied to unseen test data without data leakage.

**Key transformations:**
1. Parse mixed Weekday/Weather strings into clean categorical columns
2. Cyclical sine/cosine encoding for temporal features (Hour, Day, Month)
3. Lag features ($t-1$, $t-2$, $t-24$) aligned to prevent leakage
4. Rolling window statistics (mean, std) over past windows

In [5]:
def parse_weekday_column(series: pd.Series) -> pd.DataFrame:
    """
    Parse the mixed Weekday column.
    Expected patterns: '6 Sonnig', '6 Leicht bewölkt', '6 Bedeckt', etc.
    Extracts the numeric weekday and the text portion (if present).
    """
    weekday_num = []
    weekday_text = []
    
    for val in series.astype(str):
        # Try to extract leading number and trailing text
        match = re.match(r"^(\d+)\s*(.*)", val.strip())
        if match:
            weekday_num.append(int(match.group(1)))
            text = match.group(2).strip()
            weekday_text.append(text if text else "Unknown")
        else:
            # Fallback: try to use the entire value as categorical
            weekday_num.append(-1)
            weekday_text.append(val.strip())
    
    return pd.DataFrame({
        "weekday_num": weekday_num,
        "weekday_text": weekday_text
    })


def cyclical_encode(value: pd.Series, max_val: float) -> pd.DataFrame:
    """
    Encode a periodic feature using sine/cosine transformation.
    This preserves the cyclical nature (e.g., hour 23 is close to hour 0).
    """
    sin_vals = np.sin(2 * np.pi * value / max_val)
    cos_vals = np.cos(2 * np.pi * value / max_val)
    return sin_vals, cos_vals


def add_lag_features(df: pd.DataFrame, target_col: str = "BikeCount",
                     lags: list = None) -> pd.DataFrame:
    """
    Add lag features for the target variable.
    IMPORTANT: These are strictly backward-looking to prevent data leakage.
    """
    if lags is None:
        lags = [1, 2, 3, 6, 12, 24, 48]
    
    for lag in lags:
        df[f"lag_{lag}"] = df[target_col].shift(lag)
    
    return df


def add_rolling_features(df: pd.DataFrame, target_col: str = "BikeCount",
                         windows: list = None) -> pd.DataFrame:
    """
    Add rolling mean and std over past windows.
    Uses shift(1) to ensure we only look at past data (no current value).
    """
    if windows is None:
        windows = [3, 6, 12, 24]
    
    for w in windows:
        # shift(1) ensures we don't include the current timestep
        rolled = df[target_col].shift(1).rolling(window=w, min_periods=1)
        df[f"rolling_mean_{w}"] = rolled.mean()
        df[f"rolling_std_{w}"] = rolled.std().fillna(0)
    
    return df


def build_features(df: pd.DataFrame, is_training: bool = True,
                   target_col: str = "BikeCount") -> pd.DataFrame:
    """
    Master feature engineering function.
    Applies all transformations in sequence. Can be used for both training and inference.
    
    Parameters:
        df: Raw dataframe with original columns
        is_training: If True, target column is expected to be present
        target_col: Name of the target variable
    
    Returns:
        Fully engineered DataFrame
    """
    df = df.copy()
    
    # --- 1. Parse mixed-text Weekday column ---
    weekday_parsed = parse_weekday_column(df["Weekday"])
    df["weekday_num"] = weekday_parsed["weekday_num"]
    df["weekday_text"] = weekday_parsed["weekday_text"]
    df.drop(columns=["Weekday"], inplace=True)
    
    # --- 2. Cyclical temporal encoding ---
    # Hour: period = 24
    df["hour_sin"], df["hour_cos"] = cyclical_encode(df["Hour"], 24)
    # Day of month: period ≈ 31
    df["day_sin"], df["day_cos"] = cyclical_encode(df["Day"], 31)
    # Month: period = 12
    df["month_sin"], df["month_cos"] = cyclical_encode(df["Month"], 12)
    # Weekday: period = 7
    df["weekday_sin"], df["weekday_cos"] = cyclical_encode(df["weekday_num"], 7)
    
    # --- 3. Binary indicators ---
    df["is_weekend"] = (df["weekday_num"] >= 5).astype(int)
    df["is_rush_hour"] = df["Hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)
    df["is_night"] = df["Hour"].isin(list(range(0, 6))).astype(int)
    
    # --- 4. Lag features (only if target exists) ---
    if target_col in df.columns:
        df = add_lag_features(df, target_col)
        df = add_rolling_features(df, target_col)
    
    # --- 5. Interaction features ---
    df["temp_humidity"] = df["Temperature (°C)"] * df["Humidity (%)"]
    df["wind_rain"] = df["Wind (km/h)"] * df["Rain (mm)"]
    
    return df


print("Feature engineering functions defined.")

Feature engineering functions defined.


In [6]:
# Apply feature engineering to training data
df_feat = build_features(df_raw, is_training=True)

print(f"Engineered dataset shape: {df_feat.shape}")
print(f"\nNew columns: {df_feat.columns.tolist()}")
df_feat.head()

Engineered dataset shape: (8760, 39)

New columns: ['Month', 'Day', 'Hour', 'Weather', 'Temperature (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)', 'BikeCount', 'weekday_num', 'weekday_text', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'is_rush_hour', 'is_night', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24', 'lag_48', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24', 'temp_humidity', 'wind_rain']


,Month,Day,Hour,Weather,Temperature (°C),Humidity (%),Rain (mm),Wind (km/h),BikeCount,weekday_num,weekday_text,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos,weekday_sin,weekday_cos,is_weekend,is_rush_hour,is_night,lag_1,lag_2,lag_3,lag_6,lag_12,lag_24,lag_48,rolling_mean_3,rolling_std_3,rolling_mean_6,rolling_std_6,rolling_mean_12,rolling_std_12,rolling_mean_24,rolling_std_24,temp_humidity,wind_rain
0,1,1,0,Sunny,14.0,61.0,0.0,34.0,73.0,6,Unknown,0.000000,1.000000,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,854.0,0.0
1,1,1,1,Sunny,14.0,59.0,0.0,34.0,193.0,6,Unknown,0.258819,0.965926,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,73.0,NaN,NaN,NaN,NaN,NaN,NaN,73.000000,0.000000,73.000000,0.000000,73.000000,0.000000,73.000000,0.000000,826.0,0.0
2,1,1,2,Partly Cloudy,14.0,57.0,0.0,33.0,240.0,6,Unknown,0.500000,0.866025,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,193.0,73.0,NaN,NaN,NaN,NaN,NaN,133.000000,84.852814,133.000000,84.852814,133.000000,84.852814,133.000000,84.852814,798.0,0.0
3,1,1,3,Partly Cloudy,14.0,55.0,0.0,33.0,279.0,6,Unknown,0.707107,0.707107,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,240.0,193.0,73.0,NaN,NaN,NaN,NaN,168.666667,86.118136,168.666667,86.118136,168.666667,86.118136,168.666667,86.118136,770.0,0.0
4,1,1,4,Partly Cloudy,14.0,56.0,0.0,33.0,194.0,6,Unknown,0.866025,0.500000,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,279.0,240.0,193.0,NaN,NaN,NaN,NaN,237.333333,43.061971,196.250000,89.373281,196.250000,89.373281,196.250000,89.373281,784.0,0.0


In [7]:
# --- Prepare final feature matrix ---
# Define which columns are numeric vs. categorical for the preprocessor

TARGET_COL = "BikeCount"

# Categorical columns to one-hot encode
CAT_COLS = ["weekday_text", "Weather"]

# Columns to drop (raw temporal already encoded cyclically, or redundant)
DROP_COLS = ["Month", "Day", "Hour", "weekday_num", TARGET_COL]

# Numeric feature columns (everything else)
NUM_COLS = [c for c in df_feat.columns if c not in CAT_COLS + DROP_COLS]

print(f"Numeric features ({len(NUM_COLS)}): {NUM_COLS}")
print(f"Categorical features ({len(CAT_COLS)}): {CAT_COLS}")

Numeric features (32): ['Temperature (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'is_rush_hour', 'is_night', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24', 'lag_48', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24', 'temp_humidity', 'wind_rain']
Categorical features (2): ['weekday_text', 'Weather']


## 4. Data Split (Time-Series Ground Rules)

**Critical:** We use a strict temporal split — no shuffling. The validation set is always chronologically after the training set to simulate real-world forecasting.

In [13]:
# --- Create multi-horizon targets ---
# Horizon 1: predict t+1
df_feat["target_h1"] = df_feat[TARGET_COL].shift(-1)

# Horizon 2: predict t+1 to t+24
for h in range(1, 25):
    df_feat[f"target_h{h}"] = df_feat[TARGET_COL].shift(-h)

# Identify lag/rolling columns and target columns
lag_cols = [c for c in df_feat.columns if c.startswith("lag_") or c.startswith("rolling_")]
target_h24_cols_all = [f"target_h{h}" for h in range(1, 25)]

# Drop rows where ANY feature or target column contains NaN
# This handles: lag boundaries, target boundaries, AND missing values in original data
df_clean = df_feat.dropna(subset=NUM_COLS + CAT_COLS + lag_cols + target_h24_cols_all).copy().reset_index(drop=True)
print(f"Usable samples after removing NaN rows: {len(df_clean)}")

# Verify no NaNs remain in feature columns
assert df_clean[NUM_COLS + lag_cols].isna().sum().sum() == 0, "Features still contain NaNs!"

Usable samples after removing NaN rows: 8656


In [14]:
# --- Temporal Train/Validation Split ---
# Use last ~20% of data as validation (strictly chronological)
SPLIT_RATIO = 0.8
split_idx = int(len(df_clean) * SPLIT_RATIO)

df_train = df_clean.iloc[:split_idx].copy()
df_val = df_clean.iloc[split_idx:].copy()

print(f"Training samples: {len(df_train)}")
print(f"Validation samples: {len(df_val)}")
print(f"Train period: rows 0–{split_idx-1}")
print(f"Val period:   rows {split_idx}–{len(df_clean)-1}")

Training samples: 6924
Validation samples: 1732
Train period: rows 0–6923
Val period:   rows 6924–8655


In [15]:
# --- Build sklearn ColumnTransformer for preprocessing ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ],
    remainder="drop"
)

# Fit on training data only
X_train = preprocessor.fit_transform(df_train)
X_val = preprocessor.transform(df_val)

# Targets
y_train_h1 = df_train["target_h1"].values
y_val_h1 = df_val["target_h1"].values

# Multi-horizon targets (24 columns)
target_h24_cols = [f"target_h{h}" for h in range(1, 25)]
y_train_h24 = df_train[target_h24_cols].values
y_val_h24 = df_val[target_h24_cols].values

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_train_h1 shape: {y_train_h1.shape}")
print(f"y_train_h24 shape: {y_train_h24.shape}")

X_train shape: (6924, 65)
X_val shape:   (1732, 65)
y_train_h1 shape: (6924,)
y_train_h24 shape: (6924, 24)


## 5. Model Implementations & Training

### 5.1 Linear Model — Ridge Regression

In [16]:
# --- Horizon 1: Single-step Ridge ---
ridge_h1 = Ridge(alpha=1.0)
ridge_h1.fit(X_train, y_train_h1)

pred_ridge_h1 = ridge_h1.predict(X_val)
mse_ridge_h1 = mean_squared_error(y_val_h1, pred_ridge_h1)
print(f"Ridge — Horizon 1 (t+1) MSE: {mse_ridge_h1:.2f}")

Ridge — Horizon 1 (t+1) MSE: 23482.20


In [17]:
# --- Horizon 2: Multi-output Ridge for 24-step ahead ---
ridge_h24 = MultiOutputRegressor(Ridge(alpha=1.0))
ridge_h24.fit(X_train, y_train_h24)

pred_ridge_h24 = ridge_h24.predict(X_val)
mse_ridge_h24 = mean_squared_error(y_val_h24, pred_ridge_h24)
print(f"Ridge — Horizon 2 (t+1..t+24) MSE: {mse_ridge_h24:.2f}")

Ridge — Horizon 2 (t+1..t+24) MSE: 38994.44


### 5.2 Tree-Based Model — LightGBM

In [18]:
# --- Horizon 1: Single-step LightGBM ---
lgb_params = {
    "objective": "regression",
    "metric": "mse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "n_estimators": 1000,
    "early_stopping_rounds": 50,
    "verbose": -1,
    "random_state": RANDOM_SEED,
}

lgb_h1 = lgb.LGBMRegressor(**lgb_params)
lgb_h1.fit(
    X_train, y_train_h1,
    eval_set=[(X_val, y_val_h1)],
)

pred_lgb_h1 = lgb_h1.predict(X_val)
mse_lgb_h1 = mean_squared_error(y_val_h1, pred_lgb_h1)
print(f"\nLightGBM — Horizon 1 (t+1) MSE: {mse_lgb_h1:.2f}")


LightGBM — Horizon 1 (t+1) MSE: 6447.61


In [19]:
# --- Horizon 2: Train separate LightGBM for each of the 24 steps ---
# (Direct multi-step strategy: one model per horizon)
lgb_h24_models = []
pred_lgb_h24 = np.zeros((len(X_val), 24))

lgb_params_h24 = lgb_params.copy()
lgb_params_h24["n_estimators"] = 500  # slightly fewer for speed

for h in range(24):
    model = lgb.LGBMRegressor(**lgb_params_h24)
    model.fit(
        X_train, y_train_h24[:, h],
        eval_set=[(X_val, y_val_h24[:, h])],
    )
    lgb_h24_models.append(model)
    pred_lgb_h24[:, h] = model.predict(X_val)

mse_lgb_h24 = mean_squared_error(y_val_h24, pred_lgb_h24)
print(f"LightGBM — Horizon 2 (t+1..t+24) MSE: {mse_lgb_h24:.2f}")

LightGBM — Horizon 2 (t+1..t+24) MSE: 18524.27


### 5.3 Neural Network — PyTorch MLP

A multi-layer perceptron with residual connections, batch normalization, and dropout for regularization.

In [20]:
class BikeCountDataset(Dataset):
    """PyTorch Dataset for tabular bike count features."""
    
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class BikeCountMLP(nn.Module):
    """
    Multi-Layer Perceptron for bike count regression.
    Supports single-output (Horizon 1) or multi-output (Horizon 2).
    """
    
    def __init__(self, input_dim: int, output_dim: int = 1,
                 hidden_dims: list = None, dropout: float = 0.2):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 128, 64]
        
        layers = []
        prev_dim = input_dim
        
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


def train_mlp(model, train_loader, val_X, val_y, epochs=100, lr=1e-3,
              patience=15):
    """
    Training loop with early stopping based on validation MSE.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )
    criterion = nn.MSELoss()
    
    val_X_tensor = torch.tensor(val_X, dtype=torch.float32).to(DEVICE)
    val_y_tensor = torch.tensor(val_y, dtype=torch.float32).to(DEVICE)
    if val_y_tensor.ndim == 1:
        val_y_tensor = val_y_tensor.unsqueeze(1)
    
    best_val_mse = float("inf")
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            if y_batch.ndim == 1:
                y_batch = y_batch.unsqueeze(1)
            
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(X_batch)
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_preds = model(val_X_tensor)
            val_mse = criterion(val_preds, val_y_tensor).item()
        
        scheduler.step(val_mse)
        
        # Early stopping
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}: train_loss={epoch_loss/len(train_loader.dataset):.4f}, val_mse={val_mse:.2f}")
    
    # Restore best weights
    model.load_state_dict(best_state)
    return model, best_val_mse


print("Neural network components defined.")

Neural network components defined.


In [21]:
# --- Horizon 1: Train MLP for single-step prediction ---
BATCH_SIZE = 128
EPOCHS = 150

train_dataset_h1 = BikeCountDataset(X_train, y_train_h1)
train_loader_h1 = DataLoader(train_dataset_h1, batch_size=BATCH_SIZE, shuffle=False)
# Note: shuffle=False to preserve temporal ordering within batches

input_dim = X_train.shape[1]
mlp_h1 = BikeCountMLP(input_dim=input_dim, output_dim=1,
                       hidden_dims=[256, 128, 64]).to(DEVICE)

print(f"MLP Architecture (Horizon 1):\n{mlp_h1}")
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_h1.parameters()):,}")
print("\nTraining...")

mlp_h1, best_mse_h1 = train_mlp(
    mlp_h1, train_loader_h1, X_val, y_val_h1,
    epochs=EPOCHS, lr=1e-3, patience=20
)

# Final prediction
mlp_h1.eval()
with torch.no_grad():
    pred_mlp_h1 = mlp_h1(torch.tensor(X_val, dtype=torch.float32).to(DEVICE))
    pred_mlp_h1 = pred_mlp_h1.cpu().numpy().flatten()

mse_mlp_h1 = mean_squared_error(y_val_h1, pred_mlp_h1)
print(f"\nMLP — Horizon 1 (t+1) MSE: {mse_mlp_h1:.2f}")

MLP Architecture (Horizon 1):
BikeCountMLP(
  (network): Sequential(
    (0): Linear(in_features=65, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=64, out_features=1, bias=True)
  )
)

Total parameters: 59,009

Training...
  Epoch 20: train_loss=133086.1917, val_mse=187889.23
  Epoch 40: train_loss=25343.5878, val_mse=57468.87
  Epoch 60: train_loss=19261.4998, val_mse=43028.66
  Early stopping at epoch 70

MLP — Horizon 1 (t

In [22]:
# --- Horizon 2: Train MLP for 24-step ahead prediction ---
train_dataset_h24 = BikeCountDataset(X_train, y_train_h24)
train_loader_h24 = DataLoader(train_dataset_h24, batch_size=BATCH_SIZE, shuffle=False)

mlp_h24 = BikeCountMLP(input_dim=input_dim, output_dim=24,
                        hidden_dims=[512, 256, 128]).to(DEVICE)

print(f"MLP Architecture (Horizon 2):\n{mlp_h24}")
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_h24.parameters()):,}")
print("\nTraining...")

mlp_h24, best_mse_h24 = train_mlp(
    mlp_h24, train_loader_h24, X_val, y_val_h24,
    epochs=EPOCHS, lr=1e-3, patience=20
)

# Final prediction
mlp_h24.eval()
with torch.no_grad():
    pred_mlp_h24 = mlp_h24(torch.tensor(X_val, dtype=torch.float32).to(DEVICE))
    pred_mlp_h24 = pred_mlp_h24.cpu().numpy()

mse_mlp_h24 = mean_squared_error(y_val_h24, pred_mlp_h24)
print(f"\nMLP — Horizon 2 (t+1..t+24) MSE: {mse_mlp_h24:.2f}")

MLP Architecture (Horizon 2):
BikeCountMLP(
  (network): Sequential(
    (0): Linear(in_features=65, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=128, out_features=24, bias=True)
  )
)

Total parameters: 202,904

Training...
  Epoch 20: train_loss=144026.5254, val_mse=180401.14
  Epoch 40: train_loss=61024.2355, val_mse=110005.12
  Epoch 60: train_loss=36343.0699, val_mse=92092.46
  Epoch 80: train_loss=31175.3194, val_mse

## 6. Comparative Performance Evaluation

MSE comparison across all models and both forecast horizons.

In [23]:
# --- Results Summary ---
results = pd.DataFrame({
    "Model": ["Ridge Regression", "LightGBM", "MLP (PyTorch)"],
    "Horizon 1 MSE (t+1)": [mse_ridge_h1, mse_lgb_h1, mse_mlp_h1],
    "Horizon 2 MSE (t+1..t+24)": [mse_ridge_h24, mse_lgb_h24, mse_mlp_h24],
})

# Add rank columns
results["H1 Rank"] = results["Horizon 1 MSE (t+1)"].rank().astype(int)
results["H2 Rank"] = results["Horizon 2 MSE (t+1..t+24)"].rank().astype(int)

print("=" * 70)
print("          COMPARATIVE MODEL PERFORMANCE (Validation Set)")
print("=" * 70)
print(results.to_string(index=False))
print("=" * 70)

# Highlight best
best_h1 = results.loc[results["Horizon 1 MSE (t+1)"].idxmin(), "Model"]
best_h24 = results.loc[results["Horizon 2 MSE (t+1..t+24)"].idxmin(), "Model"]
print(f"\n🏆 Best Horizon 1: {best_h1}")
print(f"🏆 Best Horizon 2: {best_h24}")

          COMPARATIVE MODEL PERFORMANCE (Validation Set)
           Model  Horizon 1 MSE (t+1)  Horizon 2 MSE (t+1..t+24)  H1 Rank  H2 Rank
Ridge Regression         23482.204080               38994.441565        2        2
        LightGBM          6447.608249               18524.272101        1        1
   MLP (PyTorch)         38882.794146               88806.116640        3        3

🏆 Best Horizon 1: LightGBM
🏆 Best Horizon 2: LightGBM


In [24]:
# --- Optional: Per-horizon MSE breakdown for 24-step models ---
per_hour_mse = pd.DataFrame({
    "Hour Ahead": range(1, 25),
    "Ridge MSE": [mean_squared_error(y_val_h24[:, h], pred_ridge_h24[:, h]) for h in range(24)],
    "LightGBM MSE": [mean_squared_error(y_val_h24[:, h], pred_lgb_h24[:, h]) for h in range(24)],
    "MLP MSE": [mean_squared_error(y_val_h24[:, h], pred_mlp_h24[:, h]) for h in range(24)],
})

print("\nPer-Hour-Ahead MSE Breakdown:")
print(per_hour_mse.to_string(index=False))


Per-Hour-Ahead MSE Breakdown:
 Hour Ahead    Ridge MSE  LightGBM MSE       MLP MSE
          1 23482.204080   6447.608249  69573.214732
          2 27315.368557   8725.547658  64351.356569
          3 27357.821895  10742.457755  61805.049998
          4 27976.438657  11918.135000  60732.507129
          5 29961.606975  12076.549581  61178.552585
          6 33234.907685  13837.058795  64022.994643
          7 33453.989290  14667.945107  67925.320272
          8 32809.934611  16380.309351  74341.810559
          9 33939.823306  18451.837385  80835.817069
         10 39222.603497  19102.582879  87004.517243
         11 41191.499529  19751.868425  93051.635582
         12 42130.999388  21945.876694  98191.555120
         13 47757.514519  22546.444944 103540.720066
         14 47709.255343  20868.021816 107827.615823
         15 44701.169627  22575.351954 110368.862860
         16 47750.893481  22156.923166 113244.740682
         17 48316.715159  23472.200058 113556.047046
         18 448

## 7. Production Evaluation Function (Inference Block)

A self-contained function for final evaluation on an unseen test set. Accepts a new data path and a trained model pipeline, applies all preprocessing, generates predictions, and computes MSE.

In [28]:
def evaluate_final_model(
    new_csv_path: str,
    trained_model_pipeline: dict,
    horizon: int = 1,
) -> dict:
    """
    Production evaluation function for the Bike Count Estimation challenge.
    
    Parameters:
        new_csv_path: Path to the new evaluation dataset (CSV or Excel).
        trained_model_pipeline: Dictionary containing:
            - 'preprocessor': Fitted sklearn ColumnTransformer
            - 'model_h1': Trained model for Horizon 1
            - 'model_h24': Trained model for Horizon 2 (or list of models)
            - 'model_type': One of 'ridge', 'lgb', 'mlp'
            - 'num_cols': List of numeric feature column names
            - 'cat_cols': List of categorical feature column names
        horizon: 1 for single-step, 24 for multi-step
    
    Returns:
        Dictionary with 'predictions', 'mse' (if targets available), and metadata.
    """
    path = Path(new_csv_path)
    
    # --- Load data ---
    if path.suffix == ".xlsx":
        df_new = pd.read_excel(path)
    else:
        df_new = pd.read_csv(path)
    
    print(f"Loaded evaluation data: {df_new.shape}")
    
    # --- Apply feature engineering ---
    has_target = "BikeCount" in df_new.columns
    df_new_feat = build_features(df_new, is_training=has_target)
    
    # --- Create targets if available ---
    if has_target:
        if horizon == 1:
            df_new_feat["target_h1"] = df_new_feat["BikeCount"].shift(-1)
        else:
            for h in range(1, 25):
                df_new_feat[f"target_h{h}"] = df_new_feat["BikeCount"].shift(-h)
    
    # --- Remove rows with NaN in any feature or target column ---
    num_cols = trained_model_pipeline["num_cols"]
    cat_cols = trained_model_pipeline["cat_cols"]
    eval_lag_cols = [c for c in df_new_feat.columns if c.startswith("lag_") or c.startswith("rolling_")]
    
    dropna_cols = num_cols + cat_cols + eval_lag_cols
    if has_target:
        if horizon == 1:
            dropna_cols += ["target_h1"]
        else:
            dropna_cols += [f"target_h{h}" for h in range(1, 25)]
    # Only use columns that actually exist in df_new_feat
    dropna_cols = [c for c in dropna_cols if c in df_new_feat.columns]
    
    df_eval = df_new_feat.dropna(subset=dropna_cols).copy().reset_index(drop=True)
    
    # --- Transform features ---
    pre = trained_model_pipeline["preprocessor"]
    X_eval = pre.transform(df_eval)
    
    # --- Generate predictions ---
    model_type = trained_model_pipeline["model_type"]
    
    if horizon == 1:
        model = trained_model_pipeline["model_h1"]
        if model_type == "mlp":
            model.eval()
            with torch.no_grad():
                X_tensor = torch.tensor(X_eval, dtype=torch.float32).to(DEVICE)
                predictions = model(X_tensor).cpu().numpy().flatten()
        else:
            predictions = model.predict(X_eval)
    else:
        model = trained_model_pipeline["model_h24"]
        if model_type == "mlp":
            model.eval()
            with torch.no_grad():
                X_tensor = torch.tensor(X_eval, dtype=torch.float32).to(DEVICE)
                predictions = model(X_tensor).cpu().numpy()
        elif model_type == "lgb":
            # List of 24 models
            predictions = np.column_stack([
                m.predict(X_eval) for m in model
            ])
        else:
            predictions = model.predict(X_eval)
    
    # --- Compute MSE if targets are available ---
    result = {"predictions": predictions, "n_samples": len(df_eval)}
    
    if has_target:
        if horizon == 1:
            y_true = df_eval["target_h1"].values
        else:
            target_cols = [f"target_h{h}" for h in range(1, 25)]
            y_true = df_eval[target_cols].values
        
        mse = mean_squared_error(y_true, predictions)
        result["mse"] = mse
        print(f"\n{'='*50}")
        print(f"  FINAL EVALUATION MSE (Horizon {horizon}): {mse:.4f}")
        print(f"{'='*50}")
    else:
        print("No target column found — returning predictions only.")
    
    return result


print("evaluate_final_model() defined and ready.")

evaluate_final_model() defined and ready.


In [29]:
# --- Example usage: Package the best model into a pipeline dict ---

# LightGBM pipeline (typically strongest baseline for tabular data)
lgb_pipeline = {
    "preprocessor": preprocessor,
    "model_h1": lgb_h1,
    "model_h24": lgb_h24_models,
    "model_type": "lgb",
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
}

# Ridge pipeline
ridge_pipeline = {
    "preprocessor": preprocessor,
    "model_h1": ridge_h1,
    "model_h24": ridge_h24,
    "model_type": "ridge",
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
}

# MLP pipeline
mlp_pipeline = {
    "preprocessor": preprocessor,
    "model_h1": mlp_h1,
    "model_h24": mlp_h24,
    "model_type": "mlp",
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
}

print("Model pipelines packaged. Ready for final evaluation.")
print("\nUsage:")
print('  result = evaluate_final_model("path/to/test.csv", lgb_pipeline, horizon=1)')
print('  result = evaluate_final_model("path/to/test.csv", lgb_pipeline, horizon=24)')

Model pipelines packaged. Ready for final evaluation.

Usage:
  result = evaluate_final_model("path/to/test.csv", lgb_pipeline, horizon=1)
  result = evaluate_final_model("path/to/test.csv", lgb_pipeline, horizon=24)


In [30]:
# --- Validate the evaluation function on training data (sanity check) ---
print("Sanity check: evaluating LightGBM pipeline on training file...\n")
sanity_result = evaluate_final_model(
    str(TRAIN_DATA_PATH),
    lgb_pipeline,
    horizon=1
)

Sanity check: evaluating LightGBM pipeline on training file...

Loaded evaluation data: (8760, 10)

  FINAL EVALUATION MSE (Horizon 1): 2083.4671


## 8. Cross-Validation with TimeSeriesSplit (Optional Robustness Check)

In [31]:
# --- TimeSeriesSplit CV for LightGBM (Horizon 1) ---
# Provides a more robust estimate than a single temporal split

tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

# Use the full clean dataset for CV
X_full = preprocessor.fit_transform(df_clean)
y_full_h1 = df_clean["target_h1"].values

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_vl = X_full[train_idx], X_full[val_idx]
    y_tr, y_vl = y_full_h1[train_idx], y_full_h1[val_idx]
    
    model_cv = lgb.LGBMRegressor(**lgb_params)
    model_cv.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)])
    
    preds_cv = model_cv.predict(X_vl)
    fold_mse = mean_squared_error(y_vl, preds_cv)
    cv_scores.append(fold_mse)
    print(f"  Fold {fold+1}: MSE = {fold_mse:.2f}")

print(f"\nTimeSeriesSplit CV — Mean MSE: {np.mean(cv_scores):.2f} ± {np.std(cv_scores):.2f}")

  Fold 1: MSE = 12045.39
  Fold 2: MSE = 11815.21
  Fold 3: MSE = 8745.31
  Fold 4: MSE = 8281.10
  Fold 5: MSE = 5602.62

TimeSeriesSplit CV — Mean MSE: 9297.92 ± 2403.33


---
## Summary

This notebook implements a complete time-series forecasting pipeline for hourly bike counts:

1. **Feature Engineering:** Cyclical encodings, lag/rolling features, interaction terms, robust text parsing
2. **Models:** Ridge (linear), LightGBM (tree-based), MLP (neural network)
3. **Horizons:** t+1 single-step and t+1..t+24 multi-step
4. **Validation:** Strict temporal split + TimeSeriesSplit CV
5. **Production:** `evaluate_final_model()` for seamless inference on new data

**Next steps:** Hyperparameter tuning, ensemble methods, LSTM/Transformer architectures.